# Extensive Random Forest Grid Search with MLflow

This notebook uses the same loan approval dataset and runs a broad, two-stage GridSearchCV for Random Forest.

It logs all experiments to a new MLflow experiment on DagsHub.

In [7]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import dagshub
import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [8]:
data_path = '../data/raw/loan_approval_dataset.csv'
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()

print('Dataset shape:', df.shape)
df.head()

Dataset shape: (4269, 13)


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [9]:
target_col = 'loan_status'
drop_cols = [c for c in ['loan_id'] if c in df.columns]

X = df.drop(columns=[target_col] + drop_cols)
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print('Numeric columns:', numeric_features)
print('Categorical columns:', categorical_features)

Numeric columns: ['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']
Categorical columns: ['education', 'self_employed']


In [10]:
mlflow.set_tracking_uri('https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow')
dagshub.init(repo_owner='kevinsangani988', repo_name='Capstone-MLops', mlflow=True)

mlflow.set_experiment('loan_approval_random_forest_extensive_gridsearch')

Initialized MLflow to track repo "kevinsangani988/Capstone-MLops"

Repository kevinsangani988/Capstone-MLops initialized!

<Experiment: artifact_location='mlflow-artifacts:/3381165d01104cb9bba918c31343241a', creation_time=1773499148817, experiment_id='2', last_update_time=1773499148817, lifecycle_stage='active', name='loan_approval_random_forest_extensive_gridsearch', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [11]:
FULL_SEARCH = False
QUICK_RUN = True

CV_FOLDS = 3 if QUICK_RUN else 5

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

if FULL_SEARCH:
    coarse_param_grid = [
        {
            'model__n_estimators': [200, 400, 600, 800],
            'model__criterion': ['gini', 'entropy', 'log_loss'],
            'model__max_depth': [None, 10, 20, 30, 40, 60],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__max_features': ['sqrt', 'log2', None],
            'model__bootstrap': [True],
            'model__class_weight': [None, 'balanced'],
            'model__max_samples': [0.7, 0.9, None]
        },
        {
            'model__n_estimators': [200, 400, 600, 800],
            'model__criterion': ['gini', 'entropy', 'log_loss'],
            'model__max_depth': [None, 10, 20, 30, 40, 60],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__max_features': ['sqrt', 'log2', None],
            'model__bootstrap': [False],
            'model__class_weight': [None, 'balanced']
        }
    ]
elif QUICK_RUN:
    coarse_param_grid = [
        {
            'model__n_estimators': [200, 400],
            'model__criterion': ['gini'],
            'model__max_depth': [None, 20],
            'model__min_samples_split': [2, 5],
            'model__min_samples_leaf': [1, 2],
            'model__max_features': ['sqrt'],
            'model__bootstrap': [True],
            'model__class_weight': [None, 'balanced'],
            'model__max_samples': [0.8]
        }
    ]
else:
    coarse_param_grid = [
        {
            'model__n_estimators': [200, 400, 600],
            'model__criterion': ['gini', 'entropy', 'log_loss'],
            'model__max_depth': [None, 10, 20, 30],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__max_features': ['sqrt', 'log2'],
            'model__bootstrap': [True],
            'model__class_weight': [None, 'balanced'],
            'model__max_samples': [0.8, None]
        },
        {
            'model__n_estimators': [200, 400, 600],
            'model__criterion': ['gini', 'entropy', 'log_loss'],
            'model__max_depth': [None, 10, 20, 30],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__max_features': ['sqrt', 'log2'],
            'model__bootstrap': [False],
            'model__class_weight': [None, 'balanced']
        }
    ]

print('FULL_SEARCH:', FULL_SEARCH)
print('QUICK_RUN:', QUICK_RUN)
print('CV_FOLDS:', CV_FOLDS)

FULL_SEARCH: False
QUICK_RUN: True
CV_FOLDS: 3


In [12]:
with mlflow.start_run(run_name='RF_Extensive_GridSearch'):
    mlflow.log_param('full_search', FULL_SEARCH)
    mlflow.log_param('quick_run', QUICK_RUN)
    mlflow.log_param('cv_folds', CV_FOLDS)

    # Stage 1: broad search
    with mlflow.start_run(run_name='RF_Coarse_Search', nested=True):
        coarse_grid = GridSearchCV(
            estimator=rf_pipeline,
            param_grid=coarse_param_grid,
            cv=CV_FOLDS,
            scoring='accuracy',
            n_jobs=-1,
            verbose=1
        )

        coarse_grid.fit(X_train, y_train)

        coarse_best_params = coarse_grid.best_params_
        coarse_best_cv = float(coarse_grid.best_score_)

        mlflow.log_metric('coarse_best_cv_accuracy', coarse_best_cv)
        mlflow.log_params({
            f'coarse_{k.replace("model__", "")}': (v if isinstance(v, (int, float, str, bool)) else str(v))
            for k, v in coarse_best_params.items()
        })

    # Stage 2: refined search around best coarse values
    best_n_estimators = int(coarse_best_params['model__n_estimators'])
    best_max_depth = coarse_best_params['model__max_depth']
    best_min_samples_split = int(coarse_best_params['model__min_samples_split'])
    best_min_samples_leaf = int(coarse_best_params['model__min_samples_leaf'])

    estimator_step = 100 if QUICK_RUN else 200
    refined_n_estimators = sorted(list(set([
        max(100, best_n_estimators - estimator_step),
        best_n_estimators,
        best_n_estimators + estimator_step
    ])))

    if best_max_depth is None:
        refined_max_depth = [None, 20] if QUICK_RUN else [None, 20, 30, 40, 60]
    else:
        depth_step = 5 if QUICK_RUN else 10
        refined_max_depth = sorted(list(set([
            max(5, best_max_depth - depth_step),
            best_max_depth,
            best_max_depth + depth_step
        ])))

    refined_param_grid = [
        {
            'model__n_estimators': refined_n_estimators,
            'model__criterion': [coarse_best_params['model__criterion']],
            'model__max_depth': refined_max_depth,
            'model__min_samples_split': sorted(list(set([2, best_min_samples_split])) if QUICK_RUN else list(set([2, best_min_samples_split, best_min_samples_split + 2]))),
            'model__min_samples_leaf': sorted(list(set([1, best_min_samples_leaf])) if QUICK_RUN else list(set([1, best_min_samples_leaf, best_min_samples_leaf + 1]))),
            'model__max_features': [coarse_best_params['model__max_features']],
            'model__bootstrap': [coarse_best_params['model__bootstrap']],
            'model__class_weight': [coarse_best_params['model__class_weight']]
        }
    ]

    if coarse_best_params['model__bootstrap']:
        refined_param_grid[0]['model__max_samples'] = (
            [coarse_best_params.get('model__max_samples', 0.8), 0.8]
            if QUICK_RUN
            else [
                coarse_best_params.get('model__max_samples', None),
                0.7,
                0.9,
                None
            ]
        )

    with mlflow.start_run(run_name='RF_Refined_Search', nested=True):
        refined_grid = GridSearchCV(
            estimator=rf_pipeline,
            param_grid=refined_param_grid,
            cv=CV_FOLDS,
            scoring='accuracy',
            n_jobs=-1,
            verbose=1
        )

        refined_grid.fit(X_train, y_train)

        best_model = refined_grid.best_estimator_
        best_params = refined_grid.best_params_
        best_cv_accuracy = float(refined_grid.best_score_)

        y_pred = best_model.predict(X_test)
        test_accuracy = float(accuracy_score(y_test, y_pred))

        mlflow.log_metric('refined_best_cv_accuracy', best_cv_accuracy)
        mlflow.log_metric('test_accuracy', test_accuracy)

        mlflow.log_params({
            f'best_{k.replace("model__", "")}': (v if isinstance(v, (int, float, str, bool)) else str(v))
            for k, v in best_params.items()
        })

        mlflow.sklearn.log_model(best_model, name='best_random_forest_model')

    # Parent run summary
    mlflow.log_metric('final_cv_accuracy', best_cv_accuracy)
    mlflow.log_metric('final_test_accuracy', test_accuracy)

print('Best CV Accuracy:', best_cv_accuracy)
print('Best Test Accuracy:', test_accuracy)
print('Best Params:', best_params)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
🏃 View run RF_Coarse_Search at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/2/runs/6161fcd73b9e46d7a3c2568205970a66
🧪 View experiment at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/2
Fitting 3 folds for each of 24 candidates, totalling 72 fits
🏃 View run RF_Refined_Search at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/2/runs/f11f0ba558d44584b4050a0da0dcc7ea
🧪 View experiment at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/2
🏃 View run RF_Extensive_GridSearch at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/2/runs/7bd14da4833547b78de37f07c38bc98e
🧪 View experiment at: https://dagshub.com/kevinsangani988/Capstone-MLops.mlflow/#/experiments/2
Best CV Accuracy: 0.980088444369695
Best Test Accuracy: 0.9836065573770492
Best Params: {'model__bootstrap': True, 'model__class_weight': None, 'model

In [13]:
y_pred = best_model.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

    Approved       0.98      0.99      0.99       531
    Rejected       0.99      0.97      0.98       323

    accuracy                           0.98       854
   macro avg       0.99      0.98      0.98       854
weighted avg       0.98      0.98      0.98       854

Confusion Matrix:
[[528   3]
 [ 11 312]]
